# RQ2 — GSV vs Shapley-Lorenz Analysis (Dataset A)

**Nội dung:**
- Global Shapley Values (SHAP) cho LR và RF
- Shapley-Lorenz decomposition
- L1 normalize + rank 7 explanation groups
- RQ2 comparison metrics (Spearman, Kendall, Top-k Jaccard, MARD)
- Paired Bootstrap CI cho RQ2
- Tables 11.2, 11.3 + Contribution charts

> ⚠️ **Chạy trước:** `LogisticRegression.ipynb` và `RandomForest.ipynb`

In [ ]:
import sys, time, warnings, joblib
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.path import DATA_DIR

DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "datasetA"
RESULTS_DIR = DATASET_DIR / "results"
TABLES_DIR  = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
MODELS_DIR  = RESULTS_DIR / "models"
for d in (TABLES_DIR, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_COL   = "action_taken"
FEATURE_COLS = ["applicant_sex","income","loan_amount","loan_purpose","lien_status","loan_type","derived_race"]
NUM_COLS     = ["income", "loan_amount"]
GROUP_NAMES  = ["applicant_race","applicant_sex","applicant_income","loan_amount","loan_purpose","lien_status","loan_type"]

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_STATE)
print(f"Project root : {PROJECT_ROOT}")
print(f"Results dir  : {RESULTS_DIR}")

## 0. Load Models & Data

In [ ]:
# Load trained models + metadata
lr_model = joblib.load(MODELS_DIR / "lr_model.pkl")
rf_model = joblib.load(MODELS_DIR / "rf_model.pkl")
scaler   = joblib.load(MODELS_DIR / "scaler.pkl")
lr_meta  = joblib.load(MODELS_DIR / "lr_meta.pkl")
rf_meta  = joblib.load(MODELS_DIR / "rf_meta.pkl")

lr_threshold = lr_meta["threshold"]
rf_threshold = rf_meta["threshold"]
lr_train_time = lr_meta["train_time"]
rf_train_time = rf_meta["train_time"]
lr_infer_time = lr_meta["infer_time"]
rf_infer_time = rf_meta["infer_time"]

print(f"LR threshold: {lr_threshold}")
print(f"RF threshold: {rf_threshold}")

# Load test data + scale
def load_split(name):
    df = pd.read_csv(DATASET_DIR / name / f"{name}.csv")
    return df[FEATURE_COLS].copy(), df[TARGET_COL].copy()

X_test, y_test_s = load_split("test")
X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])
y_test = y_test_s.values

# Probabilities on test set
lr_prob_test = lr_model.predict_proba(X_test)[:, 1]
rf_prob_test = rf_model.predict_proba(X_test)[:, 1]
print(f"Test set size: {len(y_test):,}")

---
# PART B — RQ2: GSV vs Shapley-Lorenz (Mục 7–10)

## 1. Global Shapley Values — SHAP (Mục 7)

In [ ]:
import shap

# ── SHAP cho Logistic Regression (LinearExplainer) ──
print("Computing SHAP for LR ...")
t0 = time.perf_counter()

# Load training set cho background
X_train_bg, _ = load_split("train")
X_train_bg[NUM_COLS] = scaler.transform(X_train_bg[NUM_COLS])

explainer_lr = shap.LinearExplainer(lr_model, X_train_bg)
shap_vals_lr = explainer_lr.shap_values(X_test)
if hasattr(shap_vals_lr, "values"):
    shap_vals_lr = shap_vals_lr.values
gsv_lr_time = time.perf_counter() - t0
print(f"  Done ({gsv_lr_time:.1f}s) — shape: {shap_vals_lr.shape}")

In [ ]:
# ── SHAP cho Random Forest (TreeExplainer) ──
print("Computing SHAP for RF ...")
t0 = time.perf_counter()
explainer_rf = shap.TreeExplainer(rf_model)
shap_raw_rf  = explainer_rf.shap_values(X_test)

# Handle different SHAP output formats
if isinstance(shap_raw_rf, list):
    shap_vals_rf = shap_raw_rf[1]          # class 1 = Approved
elif hasattr(shap_raw_rf, "values"):
    shap_vals_rf = shap_raw_rf.values
elif shap_raw_rf.ndim == 3:
    shap_vals_rf = shap_raw_rf[:, :, 1]
else:
    shap_vals_rf = shap_raw_rf

gsv_rf_time = time.perf_counter() - t0
print(f"  Done ({gsv_rf_time:.1f}s) — shape: {shap_vals_rf.shape}")

### SHAP Summary Plots

In [ ]:
# ── SHAP Bar Plot — LR ──
fig, ax = plt.subplots(figsize=(10, 5))
shap.summary_plot(shap_vals_lr, X_test, plot_type="bar", show=False,
                  feature_names=FEATURE_COLS)
plt.title("SHAP Feature Importance — Logistic Regression")
plt.tight_layout()
fig = plt.gcf()
fig.savefig(FIGURES_DIR / "shap_bar_lr.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved → {FIGURES_DIR / 'shap_bar_lr.png'}")

In [ ]:
# ── SHAP Bar Plot — RF ──
fig, ax = plt.subplots(figsize=(10, 5))
shap.summary_plot(shap_vals_rf, X_test, plot_type="bar", show=False,
                  feature_names=FEATURE_COLS)
plt.title("SHAP Feature Importance — Random Forest")
plt.tight_layout()
fig = plt.gcf()
fig.savefig(FIGURES_DIR / "shap_bar_rf.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved → {FIGURES_DIR / 'shap_bar_rf.png'}")

In [ ]:
# ── SHAP Beeswarm — LR ──
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_vals_lr, X_test, show=False,
                  feature_names=FEATURE_COLS)
plt.title("SHAP Beeswarm — Logistic Regression")
plt.tight_layout()
fig = plt.gcf()
fig.savefig(FIGURES_DIR / "shap_beeswarm_lr.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved → {FIGURES_DIR / 'shap_beeswarm_lr.png'}")

In [ ]:
# ── SHAP Beeswarm — RF ──
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_vals_rf, X_test, show=False,
                  feature_names=FEATURE_COLS)
plt.title("SHAP Beeswarm — Random Forest")
plt.tight_layout()
fig = plt.gcf()
fig.savefig(FIGURES_DIR / "shap_beeswarm_rf.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved → {FIGURES_DIR / 'shap_beeswarm_rf.png'}")

## 2. Shapley-Lorenz (Mục 7)

In [ ]:
import numpy as np
np.int = getattr(np, "int", int)  # Fix NumPy compatibility

from shapley_lz.explainer.shapley_lz import ShapleyLorenzShare

SL_SAMPLE_SIZE = 50

rng = np.random.default_rng(RANDOM_STATE)
sl_idx = rng.choice(len(X_test), size=min(SL_SAMPLE_SIZE, len(X_test)), replace=False)
X_sl = X_test.iloc[sl_idx].values
y_sl = y_test[sl_idx]

def _lr_proba(X): return lr_model.predict_proba(X)[:, 1]
def _rf_proba(X): return rf_model.predict_proba(X)[:, 1]

# ── SL cho LR ──
print("Computing Shapley-Lorenz for LR ...")
t0 = time.perf_counter()
sl_lr_obj = ShapleyLorenzShare(_lr_proba, X_sl, y_sl)
sl_res_lr = sl_lr_obj.shapleyLorenz_val(X_sl, y_sl)
sl_vals_lr = np.asarray(sl_res_lr[:, 1], dtype=float)
sl_lr_time = time.perf_counter() - t0
print(f"  Done ({sl_lr_time:.1f}s)")

# ── SL cho RF ──
print("Computing Shapley-Lorenz for RF ...")
t0 = time.perf_counter()
sl_rf_obj = ShapleyLorenzShare(_rf_proba, X_sl, y_sl)
sl_res_rf = sl_rf_obj.shapleyLorenz_val(X_sl, y_sl)
sl_vals_rf = np.asarray(sl_res_rf[:, 1], dtype=float)
sl_rf_time = time.perf_counter() - t0
print(f"  Done ({sl_rf_time:.1f}s)")

## 3. Chuẩn hóa & Xếp hạng (Mục 8)

In [ ]:
def l1_normalize(c):
    """w_j = |c_j| / Σ|c_k|"""
    absv = np.abs(c)
    total = absv.sum()
    return absv / total if total > 0 else np.ones_like(absv) / len(absv)

def rank_desc(w):
    """Rank 1 = largest w."""
    order = np.argsort(-w)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, len(w) + 1)
    return ranks

# GSV: mean signed SHAP → |c_j| / Σ|c_k| → rank
c_gsv_lr = shap_vals_lr.mean(axis=0)
w_gsv_lr = l1_normalize(c_gsv_lr); r_gsv_lr = rank_desc(w_gsv_lr)

c_gsv_rf = shap_vals_rf.mean(axis=0)
w_gsv_rf = l1_normalize(c_gsv_rf); r_gsv_rf = rank_desc(w_gsv_rf)

# SL: đã là global → normalize → rank
w_sl_lr = l1_normalize(sl_vals_lr); r_sl_lr = rank_desc(w_sl_lr)
w_sl_rf = l1_normalize(sl_vals_rf); r_sl_rf = rank_desc(w_sl_rf)

# Display
for label, w, r in [
    ("LR–GSV", w_gsv_lr, r_gsv_lr), ("LR–SL", w_sl_lr, r_sl_lr),
    ("RF–GSV", w_gsv_rf, r_gsv_rf), ("RF–SL", w_sl_rf, r_sl_rf),
]:
    print(f"\n── {label} ──")
    for j, g in enumerate(GROUP_NAMES):
        print(f"  {g:<20s}  w={w[j]:.4f}  rank={int(r[j])}")

## 4. RQ2 Metrics (Mục 9)

In [ ]:
from scipy.stats import spearmanr, kendalltau

TOP_K_VALUES = [1, 2, 3]

def topk_jaccard(ranks_a, ranks_b, k):
    top_a = set(np.where(ranks_a <= k)[0])
    top_b = set(np.where(ranks_b <= k)[0])
    if not top_a and not top_b: return 1.0
    return len(top_a & top_b) / len(top_a | top_b)

def compute_rq2_metrics(r_gsv, r_sl):
    rho, _ = spearmanr(r_gsv, r_sl)
    tau, _ = kendalltau(r_gsv, r_sl)
    mard   = np.abs(r_gsv - r_sl).mean()
    out = {"Spearman ρ": rho, "Kendall τ_b": tau, "MARD": mard}
    for k in TOP_K_VALUES:
        out[f"Jaccard Top-{k}"] = topk_jaccard(r_gsv, r_sl, k)
    return out

rq2_lr = compute_rq2_metrics(r_gsv_lr, r_sl_lr)
rq2_rf = compute_rq2_metrics(r_gsv_rf, r_sl_rf)

for name, met in [("LR", rq2_lr), ("RF", rq2_rf)]:
    print(f"\n═══ {name} ═══")
    for k, v in met.items():
        print(f"  {k:<20s}: {v:.4f}")

## 5. Paired Bootstrap RQ2 (Mục 10)

1000 bootstrap resamples trên explanation observations.
- **GSV**: resample SHAP rows → recompute mean → normalize → rank
- **SL**: recompute trên bootstrap sample

In [ ]:
N_BOOTSTRAP = 5
rq2_metric_keys = list(rq2_lr.keys())

# SHAP values restricted to SL sample (paired)
shap_lr_sl = shap_vals_lr[sl_idx]
shap_rf_sl = shap_vals_rf[sl_idx]

n_sl = len(sl_idx)
boot_idx_rq2 = rng.integers(0, n_sl, size=(N_BOOTSTRAP, n_sl))

lr_rq2_boot = {k: [] for k in rq2_metric_keys}
rf_rq2_boot = {k: [] for k in rq2_metric_keys}
lr_topk_freq = {k: np.zeros(7) for k in TOP_K_VALUES}
rf_topk_freq = {k: np.zeros(7) for k in TOP_K_VALUES}

print(f"Running {N_BOOTSTRAP:,} bootstrap iterations for RQ2 (GSV vs SL)...")
t0 = time.perf_counter()

for b in range(N_BOOTSTRAP):
    idx = boot_idx_rq2[b]

    # GSV bootstrap: resample SHAP rows → normalize → rank
    w_lr_b = l1_normalize(shap_lr_sl[idx].mean(axis=0))
    r_lr_gsv_b = rank_desc(w_lr_b)
    w_rf_b = l1_normalize(shap_rf_sl[idx].mean(axis=0))
    r_rf_gsv_b = rank_desc(w_rf_b)

    # SL bootstrap: recompute on resampled data
    X_b = X_sl[idx]; y_b = y_sl[idx]
    try:
        sl_lr_b = np.asarray(ShapleyLorenzShare(_lr_proba, X_b, y_b).shapleyLorenz_val(X_b, y_b)[:, 1], dtype=float)
    except Exception:
        sl_lr_b = sl_vals_lr
    try:
        sl_rf_b = np.asarray(ShapleyLorenzShare(_rf_proba, X_b, y_b).shapleyLorenz_val(X_b, y_b)[:, 1], dtype=float)
    except Exception:
        sl_rf_b = sl_vals_rf

    r_lr_sl_b = rank_desc(l1_normalize(sl_lr_b))
    r_rf_sl_b = rank_desc(l1_normalize(sl_rf_b))

    # Metrics
    m_lr = compute_rq2_metrics(r_lr_gsv_b, r_lr_sl_b)
    m_rf = compute_rq2_metrics(r_rf_gsv_b, r_rf_sl_b)
    for k in rq2_metric_keys:
        lr_rq2_boot[k].append(m_lr[k])
        rf_rq2_boot[k].append(m_rf[k])

    # Top-k frequency
    for kv in TOP_K_VALUES:
        lr_topk_freq[kv] += (r_lr_gsv_b <= kv).astype(int)
        rf_topk_freq[kv] += (r_rf_gsv_b <= kv).astype(int)
    if (b + 1) % 50 == 0:
        gc.collect()
elapsed_rq2_boot = time.perf_counter() - t0

def ci(arr):
    return (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))

lr_rq2_ci = {k: ci(lr_rq2_boot[k]) for k in rq2_metric_keys}
rf_rq2_ci = {k: ci(rf_rq2_boot[k]) for k in rq2_metric_keys}

for kv in TOP_K_VALUES:
    lr_topk_freq[kv] /= N_BOOTSTRAP
    rf_topk_freq[kv] /= N_BOOTSTRAP

print(f"✓ RQ2 Bootstrap completed in {elapsed_rq2_boot:.2f}s")

---
# PART C — Output bắt buộc (Mục 11)

### Table 11.2 — Contribution Table

In [ ]:
rows = []
for m_name, method, w, r in [
    ("Logistic Regression", "GSV", w_gsv_lr, r_gsv_lr),
    ("Logistic Regression", "SL",  w_sl_lr,  r_sl_lr),
    ("Random Forest",       "GSV", w_gsv_rf, r_gsv_rf),
    ("Random Forest",       "SL",  w_sl_rf,  r_sl_rf),
]:
    for j, g in enumerate(GROUP_NAMES):
        rows.append({"Model": m_name, "Method": method, "Group": g,
                      "Normalized Contribution (w_j)": round(w[j], 6), "Rank": int(r[j])})

df_contrib = pd.DataFrame(rows)
df_contrib.to_csv(TABLES_DIR / "table_11_2_rq2_contributions.csv", index=False)
print(df_contrib.to_string(index=False))
print(f"\n✓ Saved → {TABLES_DIR / 'table_11_2_rq2_contributions.csv'}")

### Table 11.3 — RQ2 Comparison Table

In [ ]:
rows = []
for m_name, met, ci_d, gt, st, tkf in [
    ("Logistic Regression", rq2_lr, lr_rq2_ci, gsv_lr_time, sl_lr_time, lr_topk_freq),
    ("Random Forest",       rq2_rf, rf_rq2_ci, gsv_rf_time, sl_rf_time, rf_topk_freq),
]:
    row = {"Model": m_name}
    for k in rq2_metric_keys:
        row[k] = f"{met[k]:.4f}"
        row[f"{k} 95% CI"] = f"[{ci_d[k][0]:.4f}, {ci_d[k][1]:.4f}]"
    row["GSV Runtime (s)"] = f"{gt:.1f}"
    row["SL Runtime (s)"]  = f"{st:.1f}"
    for kv in TOP_K_VALUES:
        for j, g in enumerate(GROUP_NAMES):
            row[f"Top-{kv} freq: {g}"] = f"{tkf[kv][j]:.3f}"
    rows.append(row)

df_cmp = pd.DataFrame(rows)
df_cmp.to_csv(TABLES_DIR / "table_11_3_rq2_comparison.csv", index=False)

for _, r in df_cmp.iterrows():
    print(f"\n═══ {r['Model']} ═══")
    for k in rq2_metric_keys:
        print(f"  {k:<20s}: {r[k]}  {r[k + ' 95% CI']}")
    print(f"  GSV Runtime: {r['GSV Runtime (s)']}s")
    print(f"  SL Runtime : {r['SL Runtime (s)']}s")

print(f"\n✓ Saved → {TABLES_DIR / 'table_11_3_rq2_comparison.csv'}")

### Biểu đồ so sánh Contribution (Mục 11.4)

In [ ]:
x_pos = np.arange(len(GROUP_NAMES))
width = 0.35

# ── LR: GSV vs SL ──
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x_pos - width/2, w_gsv_lr, width, label="GSV", color="steelblue")
ax.bar(x_pos + width/2, w_sl_lr,  width, label="SL",  color="coral")
ax.set_xticks(x_pos)
ax.set_xticklabels(GROUP_NAMES, rotation=45, ha="right")
ax.set_ylabel("Normalized contribution (w_j)")
ax.set_title("Normalized Contribution — Logistic Regression: GSV vs SL")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "contribution_comparison_lr.png", dpi=150)
plt.show()

# ── RF: GSV vs SL ──
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x_pos - width/2, w_gsv_rf, width, label="GSV", color="steelblue")
ax.bar(x_pos + width/2, w_sl_rf,  width, label="SL",  color="coral")
ax.set_xticks(x_pos)
ax.set_xticklabels(GROUP_NAMES, rotation=45, ha="right")
ax.set_ylabel("Normalized contribution (w_j)")
ax.set_title("Normalized Contribution — Random Forest: GSV vs SL")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "contribution_comparison_rf.png", dpi=150)
plt.show()

print(f"✓ Figures saved → {FIGURES_DIR}")